In [1]:
!pip install tensorflow opencv-python numpy pandas scikit-learn pytube

  Using cached opencv_python-4.11.0.86-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pytube-15.0.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached absl_py-2.1.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-2.5.0-py3-none-any.whl.metadata (6.1 kB)
  Using cached tensorboard-2.18.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached keras-3.8.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pytz-2025.1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.

In [2]:
import os
import json
import numpy as np
import pandas as pd
import cv2
from pytube import YouTube

In [3]:
# Load the JSON files
with open('MSASL_train.json', 'r') as f:
    train_data = json.load(f)

with open('MSASL_test.json', 'r') as f:
    test_data = json.load(f)

with open('MSASL_val.json', 'r') as f:
    val_data = json.load(f)

# Inspect the first sample in the training data
print("First training sample:")
print(train_data[0])

First training sample:
{'org_text': 'match [light-a-MATCH]', 'clean_text': 'match', 'start_time': 0.0, 'signer_id': 0, 'signer': 0, 'start': 0, 'end': 83, 'file': 'match light-a-MATCH', 'label': 830, 'height': 360.0, 'fps': 30.0, 'end_time': 2.767, 'url': 'https://www.youtube.com/watch?v=C37R_Ix8-qs', 'text': 'match', 'box': [0.05754461884498596, 0.21637457609176636, 1.0, 0.7300844192504883], 'width': 640.0}


In [4]:
# Print dataset sizes
print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Validation samples: {len(val_data)}")

# Print the first 10 classes
with open('MSASL_classes.json', 'r') as f:
    classes = json.load(f)
print("First 10 classes:")
print(classes[:10])

Training samples: 16054
Test samples: 4172
Validation samples: 5287
First 10 classes:
['hello', 'nice', 'teacher', 'eat', 'no', 'happy', 'like', 'orange', 'want', 'deaf']


In [5]:
# Create folders for videos and frames
os.makedirs('videos', exist_ok=True)
os.makedirs('frames', exist_ok=True)

print("Folders created: 'videos', 'frames'")

Folders created: 'videos', 'frames'


In [8]:
!pip install yt-dlp

   ---------------------------------------- 0.0/3.2 MB ? eta -:--:--
   ---------------------------------------- 3.2/3.2 MB 31.4 MB/s eta 0:00:00


In [5]:
from yt_dlp import YoutubeDL

def download_video(url, output_folder):
    """Download a video using yt-dlp."""
    try:
        ydl_opts = {
            'format': 'best',  # Download the best available quality
            'outtmpl': f"{output_folder}/%(id)s.%(ext)s",  # Save with video ID as filename
            'quiet': True,  # Suppress output
        }
        with YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print(f"Downloaded: {url}")
    except Exception as e:
        print(f"Failed to download {url}: {e}")

def extract_frames(video_path, output_folder, fps=5):
    """Extract frames from a video."""
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_count % fps == 0:  # Extract every nth frame
            cv2.imwrite(f"{output_folder}/frame_{frame_count}.jpg", frame)
        frame_count += 1
    cap.release()
    print(f"Extracted {frame_count // fps} frames from {video_path}")

In [6]:
# Test downloading a video
sample = train_data[0]
video_url = sample['url']
download_video(video_url, 'videos/')

Downloaded: https://www.youtube.com/watch?v=C37R_Ix8-qs


In [7]:
# Test extracting frames
video_filename = os.listdir('videos/')[0]
video_path = os.path.join('videos', video_filename)
extract_frames(video_path, 'frames/')

Extracted 0 frames from videos\.ipynb_checkpoints


## Phase 2


In [8]:
# Filter datasets for MS-ASL100
msasl100_train = [sample for sample in train_data if sample['label'] < 100]
msasl100_test = [sample for sample in test_data if sample['label'] < 100]
msasl100_val = [sample for sample in val_data if sample['label'] < 100]

# Print sizes of filtered datasets
print(f"MS-ASL100 Training samples: {len(msasl100_train)}")
print(f"MS-ASL100 Test samples: {len(msasl100_test)}")
print(f"MS-ASL100 Validation samples: {len(msasl100_val)}")

MS-ASL100 Training samples: 3790
MS-ASL100 Test samples: 757
MS-ASL100 Validation samples: 1190


In [9]:
# Create folders for MS-ASL100
os.makedirs('videos_msasl100', exist_ok=True)
os.makedirs('frames_msasl100/train', exist_ok=True)
os.makedirs('frames_msasl100/test', exist_ok=True)
os.makedirs('frames_msasl100/val', exist_ok=True)

print("Folders created: 'videos_msasl100', 'frames_msasl100/train', 'frames_msasl100/test', 'frames_msasl100/val'")

Folders created: 'videos_msasl100', 'frames_msasl100/train', 'frames_msasl100/test', 'frames_msasl100/val'


In [10]:
video_metadata = {}

for sample in msasl100_train + msasl100_test + msasl100_val:
    url = sample['url']
    if url not in video_metadata:
        video_metadata[url] = []
    video_metadata[url].append({
        'start_time': sample['start_time'],
        'end_time': sample['end_time'],
        'label': sample['label'],
        'text': sample['text'],
        'box': sample['box'],
        'signer_id': sample['signer_id'],
        'width': sample['width'],
        'height': sample['height'],
        'fps': sample['fps']
    })

In [18]:
# Download videos and save them with unique IDs
downloaded_videos = {}

for url in video_metadata:
    try:
        video_id = url.split('v=')[-1]  # Extract video ID from URL
        if video_id not in downloaded_videos:
            download_video(url, 'videos_msasl100/')
            downloaded_videos[video_id] = True
    except Exception as e:
        print(f"Failed to download {url}: {e}")

Downloaded: https://www.youtube.com/watch?v=J7tP98oDxqE    


ERROR: [youtube] 1AyT77LqJzQ: Video unavailable


Failed to download https://www.youtube.com/watch?v=1AyT77LqJzQ: ERROR: [youtube] 1AyT77LqJzQ: Video unavailable
                                                           

Downloaded: https://www.youtube.com/watch?v=CYx7qm62Zwo


ERROR: [youtube] 7y5Ye-2-ZBs: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


...

Failed to download https://www.youtube.com/watch?v=lAE4m0AFNcw: ERROR: [youtube] lAE4m0AFNcw: Video unavailable. This video is no longer available because the YouTube account associated with this video has been terminated.
                                                           

Downloaded: https://www.youtube.com/watch?v=cxXEULq9Jpc
Downloaded: www.youtube.com/watch?v=Uu2prMSD4l4            
                                                           

Downloaded: https://www.youtube.com/watch?v=1dyyaQH7m_Y


ERROR: [youtube] 7AAnD4DyOnE: Video unavailable


Failed to download www.youtube.com/watch?v=7AAnD4DyOnE: ERROR: [youtube] 7AAnD4DyOnE: Video unavailable
Downloaded: www.youtube.com/watch?v=URb48i4X52U            
Downloaded: https://www.youtube.com/watch?v=UmcAkb54I4c    


In [12]:
def extract_clip_frames(video_path, start_time, end_time, output_folder, fps=5):
    """Extract frames for a specific clip in a video."""
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    start_frame = int(start_time * fps)
    end_frame = int(end_time * fps)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    while True:
        ret, frame = cap.read()
        if not ret or frame_count > end_frame:
            break
        if frame_count >= start_frame and frame_count <= end_frame:
            cv2.imwrite(f"{output_folder}/frame_{frame_count}.jpg", frame)
        frame_count += 1
    cap.release()
    print(f"Extracted {end_frame - start_frame + 1} frames from {video_path}")

# Extract frames for all clips and organize by class
for url, metadata_list in video_metadata.items():
    video_id = url.split('v=')[-1]  # Extract video ID from URL
    video_path = os.path.join('videos_msasl100', f"{video_id}.mp4")
    for metadata in metadata_list:
        # Create a folder for each class
        class_folder = os.path.join('frames_msasl100', f"class_{metadata['label']}")
        os.makedirs(class_folder, exist_ok=True)
        
        # Create a subfolder for this specific clip (include video ID and end time)
        clip_folder = os.path.join(
            class_folder,
            f"class_{metadata['label']}_video_{video_id}_end_{metadata['end_time']}"
        )
        os.makedirs(clip_folder, exist_ok=True)
        
        # Extract frames for this clip
        extract_clip_frames(video_path, metadata['start_time'], metadata['end_time'], clip_folder)

Extracted 14 frames from videos_msasl100\J7tP98oDxqE.mp4
Extracted 14 frames from videos_msasl100\1AyT77LqJzQ.mp4
Extracted 12 frames from videos_msasl100\1AyT77LqJzQ.mp4
Extracted 15 frames from videos_msasl100\CYx7qm62Zwo.mp4
Extracted 9 frames from videos_msasl100\7y5Ye-2-ZBs.mp4
Extracted 19 frames from videos_msasl100\iJuIO4n0-VU.mp4
Extracted 6 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 12 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 7 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 6 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 16 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 23 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 9 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 33 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 15 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 7 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 19 frames from videos_msasl100\jQb9NL9_S6U.mp4
Extracted 40 frames from videos_msasl

KeyboardInterrupt: 

In [ ]:
def preprocess_frame(frame, target_size=(224, 224)):
    """Resize and normalize a frame."""
    frame = cv2.resize(frame, target_size)  # Resize to target size
    frame = frame / 255.0  # Normalize pixel values to [0, 1]
    return frame

def save_preprocessed_frames(input_folder, output_folder, target_size=(224, 224)):
    """Preprocess and save all frames in a folder."""
    os.makedirs(output_folder, exist_ok=True)
    for frame_name in os.listdir(input_folder):
        frame_path = os.path.join(input_folder, frame_name)
        frame = cv2.imread(frame_path)
        preprocessed_frame = preprocess_frame(frame, target_size)
        output_path = os.path.join(output_folder, frame_name)
        cv2.imwrite(output_path, preprocessed_frame * 255)  # Save as 8-bit image

# Preprocess and save frames for all classes
for class_folder in os.listdir('frames_msasl100'):
    class_path = os.path.join('frames_msasl100', class_folder)
    for clip_folder in os.listdir(class_path):
        clip_path = os.path.join(class_path, clip_folder)
        output_folder = os.path.join('frames_msasl100_preprocessed', class_folder, clip_folder)
        save_preprocessed_frames(clip_path, output_folder)